## Collectable 
Represents a single record or row within a collection. Provides a standardized way to access data via keys and supports natural ordering through the Comparable interface.


In [1]:
import java.util.Map;
import java.util.List;

/**
 * Collectable Interface
 * 
 * Defines a standardized data-access and comparison contract for multiple domain entities. 
 * 
 * Key Instructional Concepts:
 * - Polymorphism: Allows different data classes to be treated uniformly through this interface.
 * - Natural Ordering: Extends Comparable to enforce a default sorting mechanism.
 * - Map Projection: Exposes object fields as key-value pairs for dynamic inspection.
 * 
 * @author John Mortensen
 * @version 2.0
 */
public interface Collectable extends Comparable<Collectable> {

    /**
     * Requires the object's internal fields to be projected into a key-value map representation.
     * 
     * @return A Map containing field names as keys and their respective values.
     */
    Map<String, Object> getMap();

    /**
     * Retrieves all available field keys for this collectable entity.
     * 
     * @return A List of String keys representing the entity's attributes.
     */
    List<String> getKeys();

    /**
     * Fetches the value of a specific field dynamically by its key string.
     * 
     * @param key The attribute key to look up.
     * @return The Object value associated with the given key.
     */
    Object getValue(String key);
}

## Catalog
Acts as the data repository for the collection, functioning similarly to a database table.



In [2]:
import java.util.*;

/**
 * Catalog Class
 * 
 * Serves as a generic container for managing a collection of Collectable items.
 * 
 * Key Instructional Concepts:
 * - Generics: Uses <T extends Collectable> to ensure type safety while allowing flexibility.
 * - Encapsulation: Manages an internal list of items, providing controlled access via methods.
 * - Collection Management: Demonstrates basic CRUD-like operations (add, retrieve, display).
 * 
 * @author John Mortensen
 * @version 2.0
 */
public class Catalog<T extends Collectable> {
    private List<T> items = new ArrayList<>();

    /**
     * Adds a new item to the catalog.
     * 
     * @param item The Collectable item to be added.
     */
    public void add(T item) { items.add(item); }

    /**
     * Retrieves the full list of items currently in the catalog.
     * 
     * @return A List containing all stored items.
     */
    public List<T> getItems() { return items; }

    /**
     * Iterates through the items and prints them to the standard output.
     */
    public void display() {
        for (T item : items) {
            System.out.println(item);
        }
    }
}

## Registry
Provides polymorphic sorting and grouping across any Collectable collection.

In [3]:
import java.util.*;

/**
 * Registry Class
 * 
 * Provides advanced processing, sorting, and classification services across mixed or 
 * uniform Collectable catalogs.
 * 
 * Key Instructional Concepts:
 * - Wildcard Generics with Upper Bounds (? extends Collectable): Allows the Registry to accept 
 *   catalogs of any subtype (e.g., Catalog<Student> or Catalog<Animal>).
 * - Dynamic Field Sorting: Uses Java Comparators to sort items on-the-fly based on string-keyed 
 *   attributes rather than fixed properties.
 * - Categorized Grouping: Uses hash-based map aggregation (Map.computeIfAbsent) to partition 
 *   items into sub-lists by attribute.
 * 
 * @author John Mortensen
 * @version 2.0
 */
public class Registry {
    
    // Backing list storing polymorphic Collectable references extracted from a Catalog
    private List<Collectable> items;

    /**
     * Constructs a Registry from any Catalog containing Collectable items.
     * 
     * @param catalog A Catalog bounded by Collectable types.
     */
    public Registry(Catalog<? extends Collectable> catalog) {
        // Polymorphic copy: Upcasts specific entities (Students/Animals) to Collectable interface type
        this.items = new ArrayList<>(catalog.getItems());
    }

    /**
     * Performs natural sorting using the items' built-in compareTo implementation.
     * 
     * @return A new sorted List of Collectable items.
     */
    public List<Collectable> sortNatural() {
        List<Collectable> sorted = new ArrayList<>(items);
        Collections.sort(sorted); // Relies on compareTo() defined in domain classes
        return sorted;
    }

    /**
     * Performs alternate sorting dynamically by evaluating any valid attribute key at runtime.
     * 
     * @param key The field key to sort by (e.g., "Age", "Pathway Year").
     * @return A sorted List of Collectable items based on the requested field.
     */
    public List<Collectable> sortByField(String key) {
        List<Collectable> sorted = new ArrayList<>(items);
        
        // Lambda comparator dynamically fetching field values via getValue()
        sorted.sort((a, b) -> {
            Object valA = a.getValue(key);
            Object valB = b.getValue(key);
            
            // Safe casting check for comparable data types (Strings, Integers, etc.)
            if (valA instanceof Comparable && valB instanceof Comparable) {
                return ((Comparable) valA).compareTo(valB);
            }
            return 0; // Fallback if non-comparable
        });
        return sorted;
    }

    /**
     * Groups collectable items into categories based on a shared attribute key.
     * 
     * @param key The attribute key to group by (e.g., "Species", "Archetype").
     * @return A Map where keys are attribute values and values are lists of matching Collectables.
     */
    public Map<Object, List<Collectable>> groupBy(String key) {
        Map<Object, List<Collectable>> groupedMap = new LinkedHashMap<>();
        
        for (Collectable item : items) {
            // Extract the grouping category value (e.g., "Scrummer" or "Mammal")
            Object groupKey = item.getValue(key);
            
            // computeIfAbsent creates a new ArrayList if the key doesn't exist yet, then adds the item
            groupedMap.computeIfAbsent(groupKey, k -> new ArrayList<>()).add(item);
        }
        return groupedMap;
    }
}

## Educational Use Case

In [4]:
import java.util.*;

/**
 * Pathway Class
 * 
 * Represents a specific curriculum track or learning journey.
 * 
 * Key Instructional Concepts:
 * - Domain Modeling: Translates real-world curriculum attributes into a structured object.
 * - Data Projection: Implements `getMap()` to allow dynamic access to its fields.
 * - Natural Ordering: Implements `Comparable` to sort pathways by year.
 * 
 * @author John Mortensen
 * @version 2.0
 */
public class Pathway implements Collectable {
    private int year;
    private String projectCapability;
    private String scrumTeamCapability;
    private String problemSolving;
    private String initiative;

    public Pathway(int year, String projectCapability, String scrumTeamCapability, String problemSolving, String initiative) {
        this.year = year;
        this.projectCapability = projectCapability;
        this.scrumTeamCapability = scrumTeamCapability;
        this.problemSolving = problemSolving;
        this.initiative = initiative;
    }

    public int getYear() { return year; }
    public String getProjectCapability() { return projectCapability; }

    @Override
    public Map<String, Object> getMap() {
        Map<String, Object> map = new LinkedHashMap<>();
        map.put("Year", year);
        map.put("Project Capability", projectCapability);
        map.put("Scrum Team", scrumTeamCapability);
        map.put("Problem Solving", problemSolving);
        map.put("Initiative", initiative);
        return map;
    }

    @Override public List<String> getKeys() { return new ArrayList<>(getMap().keySet()); }
    @Override public Object getValue(String key) { return getMap().get(key); }
    @Override public int compareTo(Collectable o) { return Integer.compare(this.year, (Integer) o.getValue("Year")); }
    @Override public String toString() { return "Year " + year + " (" + projectCapability + ")"; }

    /**
     * Returns a pre-populated catalog of curriculum pathways.
     * 
     * @return A Catalog containing the four standard pathway years (0-3).
     */
    public static Catalog<Pathway> catalog() {
        Catalog<Pathway> catalog = new Catalog<>();
        catalog.add(new Pathway(0, "Game", "Solo", "Guided", "Passive"));
        catalog.add(new Pathway(1, "Project", "Pair (2-3)", "Developing", "Self-starter"));
        catalog.add(new Pathway(2, "System", "Group (~6)", "", "Creator"));
        catalog.add(new Pathway(3, "Research", "Team (~10)", "Engaging", "Driver"));
        return catalog;
    }
}

In [5]:
import java.util.*;

/**
 * Standing Class
 * 
 * Represents an academic or professional status level.
 * 
 * Key Instructional Concepts:
 * - Attribute Weighting: Uses a numeric weight to facilitate mathematical comparisons.
 * - Data Projection: Implements `getMap()` for dynamic attribute inspection.
 * - Natural Ordering: Implements `Comparable` to sort by weight.
 * 
 * @author John Mortensen
 * @version 2.0
 */
public class Standing implements Collectable {
    private String label;
    private int weight;

    public Standing(String label, int weight) {
        this.label = label;
        this.weight = weight;
    }

    public String getLabel() { return label; }
    public int getWeight() { return weight; }

    @Override
    public Map<String, Object> getMap() {
        Map<String, Object> map = new LinkedHashMap<>();
        map.put("Label", label);
        map.put("Weight", weight);
        return map;
    }

    @Override public List<String> getKeys() { return new ArrayList<>(getMap().keySet()); }
    @Override public Object getValue(String key) { return getMap().get(key); }
    @Override public int compareTo(Collectable o) { return Integer.compare(this.weight, (Integer) o.getValue("Weight")); }
    @Override public String toString() { return label + " (Weight: " + weight + ")"; }

    /**
     * Returns a pre-populated catalog of academic standings.
     * 
     * @return A Catalog containing Freshman, Sophomore, Junior, and Senior standings.
     */
    public static Catalog<Standing> catalog() {
        Catalog<Standing> catalog = new Catalog<>();
        catalog.add(new Standing("Freshman", 0));
        catalog.add(new Standing("Sophomore", 1));
        catalog.add(new Standing("Junior", 2));
        catalog.add(new Standing("Senior", 3));
        return catalog;
    }
}

In [6]:
import java.util.*;

/**
 * Student Class
 * 
 * A complex domain entity that aggregates multiple `Collectable` components.
 * 
 * Key Instructional Concepts:
 * - Composition: Demonstrates how a class can be composed of other complex objects (`Standing`, `Pathway`).
 * - Dependency Management: Shows how a parent entity relies on the state of its child components.
 * - Polymorphic Comparison: Implements `compareTo` to provide a natural sort order based on the student's name.
 * - Map Projection: Flattens hierarchical data into a single-level map for easy registry access.
 * 
 * @author John Mortensen
 * @version 2.0
 */
public class Student implements Collectable {
    private String name;
    private Standing standing;
    private Pathway pathway;
    private String archetype;

    public Student(String name, Standing standing, Pathway pathway, String archetype) {
        this.name = name;
        this.standing = standing;
        this.pathway = pathway;
        this.archetype = archetype;
    }

    public String getName() { return name; }
    public Standing getStanding() { return standing; }
    public Pathway getPathway() { return pathway; }
    public String getArchetype() { return archetype; }

    @Override
    public Map<String, Object> getMap() {
        Map<String, Object> map = new LinkedHashMap<>();
        map.put("Name", name);
        map.put("Standing", standing.getLabel());
        map.put("Standing Weight", standing.getWeight());
        map.put("Pathway Year", pathway.getYear());
        map.put("Project Capability", pathway.getProjectCapability());
        map.put("Archetype", archetype);
        return map;
    }

    @Override public List<String> getKeys() { return new ArrayList<>(getMap().keySet()); }
    @Override public Object getValue(String key) { return getMap().get(key); }

    @Override
    public int compareTo(Collectable o) {
        // Natural sorting by Name alphabetically
        return this.name.compareTo((String) o.getValue("Name"));
    }

    @Override
    public String toString() {
        return String.format("Student [Name: %s, Standing: %s, Pathway: Year %d (%s), Archetype: %s]",
                name, standing.getLabel(), pathway.getYear(), pathway.getProjectCapability(), archetype);
    }
}

In [15]:
import java.util.*;

/**
 * Students Driver Class
 * 
 * Serves as the main entry point for the demonstration, orchestrating the interaction 
 * between Catalogs, Registries, and domain entities.
 * 
 * Key Instructional Concepts:
 * - System Orchestration: Demonstrates how to initialize reference data and primary rosters.
 * - Registry Usage: Shows how a `Registry` can be used to perform complex operations on a `Catalog`.
 * - Data-Driven Testing: Uses a series of print statements to verify sorting, grouping, and display logic.
 * 
 * @author John Mortensen
 * @version 2.0
 */
public class Students {
    public static Catalog<Student> catalog() {
        // Master Catalog for Reference Data (Pathways & Standings)
        Catalog<Pathway> pathwayCatalog = Pathway.catalog();
        Catalog<Standing> standingCatalog = Standing.catalog(); 
        // Student Roster Catalog
        Catalog<Student> studentCatalog = new Catalog<>();
        studentCatalog.add(new Student("Thomas Edison", standingCatalog.getItems().get(3), pathwayCatalog.getItems().get(3), "Scrummer"));
        studentCatalog.add(new Student("Nikola Tesla", standingCatalog.getItems().get(3), pathwayCatalog.getItems().get(2), "Technologist"));
        studentCatalog.add(new Student("Grace Hopper", standingCatalog.getItems().get(2), pathwayCatalog.getItems().get(3), "Planner"));
        studentCatalog.add(new Student("Marie Curie", standingCatalog.getItems().get(2), pathwayCatalog.getItems().get(1), "Analyst"));
        studentCatalog.add(new Student("Alan Turing", standingCatalog.getItems().get(2), pathwayCatalog.getItems().get(1), "Technologist"));
        studentCatalog.add(new Student("Hedy Lamarr", standingCatalog.getItems().get(2), pathwayCatalog.getItems().get(2), "Planner"));
        studentCatalog.add(new Student("isaac Newton", standingCatalog.getItems().get(1), pathwayCatalog.getItems().get(0), "Analyst"));
        studentCatalog.add(new Student("Bill Gates", standingCatalog.getItems().get(0), pathwayCatalog.getItems().get(0), "Analyst"));
        return studentCatalog;
    }
 
    public static void main(String[] args) {
        // Initialize the student catalog
        Catalog<Student> studentCatalog = catalog();
        // Initialize the registry with the student catalog
        Registry registry = new Registry(studentCatalog);

        // View the student catalog
        System.out.println("=== 1. Student toString() & Catalog Display ===");
        studentCatalog.display();
        // Sort and View the registry naturally (by Name)
        System.out.println("\n=== 2. Natural Sorting (compareTo by Name) ===");
        registry.sortNatural().forEach(System.out::println);
        // Sort and View the registry by an alternate field (Pathway Year)
        System.out.println("\n=== 3. Alternate Sorting (by Pathway Year) ===");
        registry.sortByField("Pathway Year").forEach(System.out::println);

        // Group the registry by Archetype using groupBy functionality
        System.out.println("\n=== 4. Grouping by Archetype ===");
        Map<Object, List<Collectable>> grouped = registry.groupBy("Archetype");
        grouped.forEach((key, value) -> {
            System.out.println("Archetype: " + key);
            value.forEach(item -> System.out.println("  - " + item));
        });
    }
}
Students.main(null);

=== 1. Student toString() & Catalog Display ===
Student [Name: Thomas Edison, Standing: Senior, Pathway: Year 3 (Research), Archetype: Scrummer]
Student [Name: Nikola Tesla, Standing: Senior, Pathway: Year 2 (System), Archetype: Technologist]
Student [Name: Grace Hopper, Standing: Junior, Pathway: Year 3 (Research), Archetype: Planner]
Student [Name: Marie Curie, Standing: Junior, Pathway: Year 1 (Project), Archetype: Analyst]
Student [Name: Alan Turing, Standing: Junior, Pathway: Year 1 (Project), Archetype: Technologist]
Student [Name: Hedy Lamarr, Standing: Junior, Pathway: Year 2 (System), Archetype: Planner]
Student [Name: isaac Newton, Standing: Sophomore, Pathway: Year 0 (Game), Archetype: Analyst]
Student [Name: Bill Gates, Standing: Freshman, Pathway: Year 0 (Game), Archetype: Analyst]

=== 2. Natural Sorting (compareTo by Name) ===
Student [Name: Alan Turing, Standing: Junior, Pathway: Year 1 (Project), Archetype: Technologist]
Student [Name: Bill Gates, Standing: Freshman, P

## Animal References

In [8]:
import java.util.ArrayList;
import java.util.LinkedHashMap;
import java.util.List;
import java.util.Map;

/**
 * Animal Class
 * 
 * Represents a concrete domain entity implementing the Collectable contract. 
 * This class models simple animal records with a name, species, and age.
 * 
 * Key Instructional Concepts:
 * - Interface Implementation: Fulfills the Collectable contract for uniform processing.
 * - Natural Ordering: Implements compareTo() to define default sorting behavior (alphabetical by name).
 * - Data Projection: Implements getMap() to expose object attributes dynamically for registry mapping.
 * 
 * @author John Mortensen
 * @version 2.0
 */
public class Animal implements Collectable {
    
    private String name;
    private String species;
    private int age;

    /**
     * Constructs a new Animal record.
     * 
     * @param name The name of the animal.
     * @param species The biological species or category.
     * @param age The age of the animal in years.
     */
    public Animal(String name, String species, int age) {
        this.name = name;
        this.species = species;
        this.age = age;
    }

    public String getName() { return name; }
    public String getSpecies() { return species; }
    public int getAge() { return age; }

    /**
     * Projects internal fields into a LinkedHashMap to maintain insertion order 
     * when inspected dynamically by registries or export tools.
     */
    @Override
    public Map<String, Object> getMap() {
        Map<String, Object> map = new LinkedHashMap<>();
        map.put("Name", name);
        map.put("Species", species);
        map.put("Age", age);
        return map;
    }

    @Override
    public List<String> getKeys() {
        return new ArrayList<>(getMap().keySet());
    }

    @Override
    public Object getValue(String key) {
        return getMap().get(key);
    }

    /**
     * Defines natural ordering (Comparable interface contract).
     * Compares animals alphabetically by their name field.
     */
    @Override
    public int compareTo(Collectable o) {
        // Safe extraction and comparison of the "Name" attribute from another Collectable entity
        String otherName = (String) o.getValue("Name");
        return this.name.compareTo(otherName);
    }

    @Override
    public String toString() {
        return String.format("Animal [Name: %s, Species: %s, Age: %d]", name, species, age);
    }
}

In [11]:
/**
 * Animals Tester Class
 * 
 * Serves as an executable runner to demonstrate catalog management, natural sorting, 
 * alternate dynamic field sorting, and grouping using the shared Registry architecture.
 * 
 * Key Instructional Concepts:
 * - Polymorphism in Action: Demonstrates how a shared Registry handles Animal entities 
 *   through the Collectable interface abstraction.
 * - Separation of Concerns: Catalog handles data storage, Registry handles processing and organization.
 * 
 * @author John Mortensen
 * @version 2.0
 */
public class Animals {
    
    public static void main(String[] args) {
        
        // 1. Initialize a generic Catalog bounded to Animal entities
        Catalog<Animal> animalCatalog = new Catalog<>();
        
        // Populate the catalog with sample records (out of alphabetical order to test sorting)
        animalCatalog.add(new Animal("Zebra", "Mammal", 5));
        animalCatalog.add(new Animal("Aardvark", "Mammal", 3));
        animalCatalog.add(new Animal("Falcon", "Bird", 2));
        animalCatalog.add(new Animal("Baboon", "Mammal", 4));

        // 2. Wrap the catalog inside the Registry processor
        Registry registry = new Registry(animalCatalog);

        // 3. Demonstrate standard display utilizing polymorphic toString()
        System.out.println("=== 1. Catalog Display (Insertion Order) ===");
        animalCatalog.display();

        // 4. Demonstrate natural sorting via compareTo() (Alphabetical by Name)
        System.out.println("\n=== 2. Natural Sorting (Alphabetical by Name) ===");
        registry.sortNatural().forEach(System.out::println);

        // 5. Demonstrate alternate dynamic sorting by a non-default field key ("Age")
        System.out.println("\n=== 3. Alternate Sorting (Dynamic Sort by Age) ===");
        registry.sortByField("Age").forEach(System.out::println);

        // 6. Demonstrate data grouping/partitioning by an attribute key ("Species")
        System.out.println("\n=== 4. Categorized Grouping by Species ===");
        registry.groupBy("Species").forEach((species, list) -> {
            System.out.println("Species Group: " + species);
            list.forEach(animal -> System.out.println("    - " + animal));
        });
    }
}
Animals.main(null);

=== 1. Catalog Display (Insertion Order) ===
Animal [Name: Zebra, Species: Mammal, Age: 5]
Animal [Name: Aardvark, Species: Mammal, Age: 3]
Animal [Name: Falcon, Species: Bird, Age: 2]
Animal [Name: Baboon, Species: Mammal, Age: 4]

=== 2. Natural Sorting (Alphabetical by Name) ===
Animal [Name: Aardvark, Species: Mammal, Age: 3]
Animal [Name: Baboon, Species: Mammal, Age: 4]
Animal [Name: Falcon, Species: Bird, Age: 2]
Animal [Name: Zebra, Species: Mammal, Age: 5]

=== 3. Alternate Sorting (Dynamic Sort by Age) ===
Animal [Name: Falcon, Species: Bird, Age: 2]
Animal [Name: Aardvark, Species: Mammal, Age: 3]
Animal [Name: Baboon, Species: Mammal, Age: 4]
Animal [Name: Zebra, Species: Mammal, Age: 5]

=== 4. Categorized Grouping by Species ===
Species Group: Mammal
    - Animal [Name: Zebra, Species: Mammal, Age: 5]
    - Animal [Name: Aardvark, Species: Mammal, Age: 3]
    - Animal [Name: Baboon, Species: Mammal, Age: 4]
Species Group: Bird
    - Animal [Name: Falcon, Species: Bird, Ag